# Day 2 - Toolbox & Evaluation

> Work through the exercises in order - each follows a segment of the
> session. Cells marked `TODO` are yours to fill in; a `checks.check_ex_*`
> call tells you whether it worked. Stretch sections are optional.

## Setup

Same `coursekit` imports, plus `statsforecast` for the models and
`utilsforecast` for the metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import (HistoricAverage, Naive, RandomWalkWithDrift,
                                  SeasonalNaive)
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.losses import mae, mape, mase, rmse, rmsse

from coursekit import checks
from coursekit import datasets as D
from coursekit import leaderboard as lb
from coursekit import plotting as P

P.use_course_style()

spine = D.spine()
H = 24
train, test = D.train_test(spine, h=H)
print(f"train: {len(train)} months to {train['ds'].max().date()}")
print(f"test : {len(test)} months from {test['ds'].min().date()}")

---
# Exercise 2.1 - The benchmark floor

*Follows segment 1. 13 minutes.*

Fit all four benchmarks and look at them. Everything for the rest of the course
is measured against these.

In [ ]:
MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD"]
LABELS = {"HistoricAverage": "Mean", "Naive": "Naive",
          "SeasonalNaive": "Seasonal naive", "RWD": "Drift"}

# TODO: build a StatsForecast object with all four benchmarks and forecast H
#       months ahead from `train`, with 80% and 95% intervals, keeping fitted values.
sf = ...
fc = ...

checks.check_ex_2_1(fc, MODELS)
fc.head()

In [ ]:
# TODO: plot the last 6 years of training data, the four forecasts, and the
#       held-out actuals on one chart.

**Question.** Two of these are obviously wrong before you compute a single
metric. Which, and why?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* The **mean** method forecasts a flat line at roughly 160 for a series
currently sitting near 370 - it averages over 37 years of growth, so it is
hopeless on any trending series. The **naive** method forecasts a flat line at
the last value, which throws away the seasonality we spent all of Day 1
establishing. **Drift** at least captures the trend but still ignores season.
Only the **seasonal naive** reproduces the annual shape.

### Stretch - forecasting on a transformed scale

The spine is multiplicative. Forecast the Box-Cox transformed series, then
back-transform. Note that the naive back-transform gives you the **median**, not
the mean.

In [ ]:
# Stretch - your code here.

---
# Exercise 2.2 - Are the residuals white noise?

*Follows segment 2. 13 minutes.*

If a model's residuals still carry structure, the model has not finished.

In [ ]:
fv = sf.forecast_fitted_values()

# TODO: compute the seasonal naive's residuals, plot the three-panel
#       diagnostic, and run a Ljung-Box test at lag 24.
resid = ...
lb_pvalue = ...

print(f"mean residual : {pd.Series(resid).mean():.3f}")
print(f"Ljung-Box p   : {lb_pvalue:.3e}")

checks.check_ex_2_2(resid, lb_pvalue)

In [ ]:
# TODO: do the same for the drift method. Which of the four properties
#       (uncorrelated / zero mean / constant variance / normal) does each satisfy?

**Write your verdict.** For each method, which of the four residual properties
hold, and what does that imply?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Neither is close to white noise.

- **Uncorrelated:** fails badly for both - Ljung-Box p is effectively zero and
  the residual ACF has large spikes. There is a great deal of signal left.
- **Zero mean:** the seasonal naive's mean residual is clearly positive, because
  the series trends upward and last year's value is systematically too low. That
  is a *bias*: the forecast will be low every time.
- **Constant variance:** fails - the late-period standard deviation is several
  times the early one, because the series grew eightfold. This is exactly what
  the Box-Cox transform in 1.4 addresses.
- **Normal:** roughly, but with heavy tails.

Implication: the benchmark floor is a floor, not a model. The failures are
informative - the bias says "add a trend", the seasonal spikes say "the seasonal
shape has changed", the variance says "transform first".

---
# Exercise 2.3 - Intervals, and how much to believe them

*Follows segment 3. 15 minutes.*

In [ ]:
# TODO: plot the seasonal naive's forecast with 80% and 95% intervals against
#       the held-out actuals. (P.fan_chart wants columns mean / lo-80 / hi-80 / ...)

In [ ]:
# TODO: the 80% interval width at each horizon, and how it compares with
#       width_1 * sqrt(h).
width = ...

h = np.arange(1, len(width) + 1)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(h, width, color=P.BLUE, lw=2, label="actual width")
ax.plot(h, width.iloc[0] * np.sqrt(h), color=P.ORANGE, ls="--", lw=1.4,
        label="width_1 * sqrt(h)")
ax.set(xlabel="horizon h", ylabel="80% interval width", title="Widening with h")
ax.legend(frameon=False)
plt.show()

In [ ]:
merged = test.merge(fc, on=["unique_id", "ds"])

# TODO: what fraction of the held-out actuals fall inside the 80% interval?
#       And what is the standard error of that estimate?
coverage = ...
se = ...

print(f"nominal 80%,  measured {coverage:.1%}  +/- {1.96 * se:.1%} (95% CI)")
checks.check_ex_2_3(width, coverage, se)

**Question.** Your measured coverage came with an error bar roughly 16 points
wide. What would you have to change to measure coverage properly - and is that
what exercise 2.5 does?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* You need more scored points, and they must come from *different
origins* rather than from extending one test window (extending it just forecasts
further ahead, where the model is worse). Rolling-origin cross-validation gives
exactly that: 8 folds x 12 months = 96 scored points instead of 24, cutting the
standard error in half. That is exercise 2.5.

Note it does not fix the *other* problem - the interval formula ignores model
uncertainty - so even a well-measured coverage tends to come in under nominal.

---
# Exercise 2.4 - Scoring, and the metric that lies

*Follows segment 4. 12 minutes.*

In [ ]:
# TODO: build a table of MAE, RMSE, MAPE, MASE and RMSSE for all four models
#       on the holdout. MASE and RMSSE need seasonality=12 and train_df=train.
scores = ...

checks.check_ex_2_4(scores)
scores.round(3)

Now build the case where MAPE misleads. Construct a near-zero series and two
forecasts: one that is a little too **low**, one that is much too **high**.

In [ ]:
rng = np.random.default_rng(3)
n = 48
low = pd.DataFrame({
    "ds": pd.date_range("2020-01-01", periods=n, freq="MS"),
    "y": np.clip(rng.poisson(1.4, n).astype(float), 0.2, None),
})

# TODO: forecast A is always 2.0 units too HIGH; forecast B is always 0.15 too LOW.
#       Compute MAE and MAPE for each. Which does MAE prefer? Which does MAPE prefer?

**Rank the four benchmarks and defend the ranking.** Which metric did you use,
and why not the others?

<!--STUDENT-->
*Your answer:*

<!--SOLUTION-->
*Answer.* Seasonal naive > Naive > Drift > Mean, on MASE.

MASE, because it is scale-free (so this ranking can be compared against other
series later), it is defined even when the series touches zero, and the
benchmark is built into it - a MASE of 1.11 immediately tells you the winner is
still slightly worse than a one-step seasonal naive.

Not MAE or RMSE: correct here, but their units are millions of dollars, so they
cannot be pooled across series. Not MAPE: this series never approaches zero so
it happens to behave, but selecting on MAPE builds a habit that breaks the first
time you meet slow-moving demand.

---
# Exercise 2.5 - The harness

*Follows segment 5. 17 minutes.*

This is the exercise the rest of the course rests on. You are building the
evaluation harness that every Day 3 model gets plugged into.

In [ ]:
# TODO: rolling-origin cross-validation over the WHOLE spine:
#       8 origins, 12 months forecast each, 80% intervals.
cv = ...

print(f"folds : {cv['cutoff'].nunique()}")
print(f"scored points : {len(cv)}")
cv.head()

In [ ]:
# TODO: for each model compute, ACROSS FOLDS:
#         - mean MASE   (score each fold against its own training data)
#         - mean RMSSE
#         - empirical 80% coverage
#       Return a tidy frame with columns: model / mase / rmsse / coverage_80
summary = ...

checks.check_ex_2_5(cv, summary)
summary.round(3)

Write the results to the leaderboard. **This file is the course's running
scoreboard** - Day 3 appends to the same table.

In [ ]:
lb.reset()   # start clean; re-running this cell is safe

for _, row in summary.iterrows():
    lb.record(
        row["model"], day=2,
        mase=float(row["mase"]), rmsse=float(row["rmsse"]),
        coverage_80=float(row["coverage_80"]),
        notes="benchmark, 8-fold rolling origin",
    )

table = lb.show()
checks.check_leaderboard(table)
table.round(3)

### Stretch - how much does one window matter?

Score each fold separately and look at the spread.

In [ ]:
# Stretch - your code here.

---
## End of Day 2

You have an evaluation harness: benchmarks, residual diagnostics, intervals with
an honest error bar, scale-free metrics, and rolling-origin cross-validation.

`labs/leaderboard.csv` now holds the benchmark floor. Every model on Day 3 has
to get past it.